# Dataset Exploratory Data Analysis
Reproduces Table 1 and Figure 5 (subgroup imbalance heatmap) from the paper.

Dataset: 2,821 images | 24 intersectional subgroups | worst-case 35.47:1 imbalance

In [ ]:
import sys
sys.path.insert(0, '..')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns

# Paper Table 1 data (Gender x Ethnicity)
data = {
    'Male':   {'White': (674,516), 'Asian': (275,183), 'Black': (69,95),  'Dark': (74,69)},
    'Female': {'White': (160,310), 'Asian': (107,134), 'Black': (22,60),  'Dark': (19,54)},
}

genders     = ['Male', 'Female']
ethnicities = ['White', 'Asian', 'Black', 'Dark']

n_total = np.array([[data[g][e][0]+data[g][e][1] for e in ethnicities] for g in genders])
n_asd   = np.array([[data[g][e][0] for e in ethnicities] for g in genders])
n_nonasd = np.array([[data[g][e][1] for e in ethnicities] for g in genders])

print(f'Total samples: {n_total.sum()}')
print(f'ASD:     {n_asd.sum()}')
print(f'Non-ASD: {n_nonasd.sum()}')
print(f'Worst-case imbalance: {n_total.max()/n_total.min():.2f}:1')
print(f'Largest subgroup:  Male x White  (n={n_total[0,0]})')
print(f'Smallest subgroup: Female x Dark (n={n_total[1,3]})')

In [ ]:
# Heatmap: subgroup sample counts (Figure 5)
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

for ax, data_arr, title, cmap in zip(
    axes,
    [n_total, n_asd, n_nonasd],
    ['Total samples', 'ASD (positive)', 'Non-ASD (negative)'],
    ['Greens', 'Blues', 'Oranges'],
):
    im = ax.imshow(data_arr, cmap=cmap, aspect='auto')

    ax.set_xticks(range(len(ethnicities)))
    ax.set_xticklabels(ethnicities, fontsize=11)
    ax.set_yticks(range(len(genders)))
    ax.set_yticklabels(genders, fontsize=11)
    ax.set_title(title, fontsize=13, fontweight='bold')

    # Annotate cells
    for i in range(len(genders)):
        for j in range(len(ethnicities)):
            val = data_arr[i, j]
            color = 'white' if val > data_arr.mean() else 'black'
            ax.text(j, i, str(val), ha='center', va='center',
                    color=color, fontsize=12, fontweight='bold')

    # Red border on smallest subgroups
    for (gi, ei) in [(1, 2), (1, 3)]:
        rect = plt.Rectangle((ei-0.5, gi-0.5), 1, 1,
                              linewidth=2.5, edgecolor='red', facecolor='none')
        ax.add_patch(rect)

    plt.colorbar(im, ax=ax, shrink=0.8)

fig.suptitle(
    'Intersectional subgroup distribution (Gender × Ethnicity)\n'
    'Red border: smallest subgroups (n=22, n=73). '
    'Worst-case imbalance: 35.47:1',
    fontsize=12, y=1.02
)
plt.tight_layout()
plt.savefig('subgroup_heatmap.pdf', dpi=300, bbox_inches='tight')
plt.show()
print('Saved: subgroup_heatmap.pdf')

In [ ]:
# Imbalance ratio table
rows = []
for gi, g in enumerate(genders):
    for ei, e in enumerate(ethnicities):
        n = n_total[gi, ei]
        rows.append({'Gender': g, 'Ethnicity': e,
                     'n_total': n, 'n_ASD': n_asd[gi,ei],
                     'n_nonASD': n_nonasd[gi,ei],
                     'ratio_to_max': round(n_total.max()/n, 2)})

df = pd.DataFrame(rows).sort_values('n_total')
print('Subgroup distribution (sorted by size):')
print(df.to_string(index=False))

In [ ]:
# Data source breakdown
print('Dataset composition:')
print(f'  Kaggle (international): 2,122 images (78.2% White-presenting)')
print(f'  Vietnamese children:      699 images (collected via VN-ASDetect app)')
print(f'  Total:                  2,821 images')
print()
print('Fixed split (performed before augmentation):')
print(f'  Train: 2,258 (80%)')
print(f'  Val:     281 (10%)')
print(f'  Test:    282 (10%)')